# Day 5 · Một notebook tự kiểm từ đầu đến cuối

**Tùy chọn, không chấm điểm.** Chạy từng ô từ trên xuống bằng Shift+Enter/Runtime → Run all. Nếu dùng Colab, chỉ upload file notebook này, dán link fork công khai của bạn vào `FORK_URL` ở ô kế tiếp; notebook sẽ tải ảnh và công cụ tự kiểm từ fork. Nếu chưa push ZIP CVAT, dùng ô upload ZIP tạm phía dưới. Không có đáp án trong notebook. Nếu không dùng code, làm hoàn toàn theo `lab-guide.html`.

In [ ]:
from pathlib import Path
import importlib.util
import sys
import re
import subprocess

# Trên Colab: dán URL fork của BẠN, ví dụ https://github.com/ten-ban/Day5-Segmentation-Lab-Student
FORK_URL = ""
IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    assert re.fullmatch(r"https://github\.com/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+(?:\.git)?/?", FORK_URL), "Dán link fork GitHub vào FORK_URL rồi chạy lại ô này."
    root = Path("/content/day5-student-fork")
    if not (root / "data/manifest.json").is_file():
        subprocess.run(["git", "clone", "--depth", "1", FORK_URL, str(root)], check=True)
    else:
        existing_url = subprocess.run(["git", "-C", str(root), "remote", "get-url", "origin"], check=True, capture_output=True, text=True).stdout.strip()
        assert existing_url.rstrip("/").removesuffix(".git") == FORK_URL.rstrip("/").removesuffix(".git"), "Phiên Colab đang có fork khác. Chọn Runtime > Restart runtime rồi chạy lại."
        print("Đang dùng bản fork đã tải trong phiên Colab này; Runtime > Restart để tải lại bản mới.")
else:
    root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/manifest.json").is_file()), None)
    assert root is not None, "Mở notebook từ thư mục repo Day 5 (hoặc thư mục notebooks/)."
sys.path.insert(0, str(root / "scripts"))
from inspect_submissions import task_registry, expected_for, inspect_task
tasks = task_registry(root)
exports = root / "submissions"
exports.mkdir(exist_ok=True)
print("Repo:", root)
print("Thư mục export:", exports)


## Nếu cần, đưa ZIP CVAT vào Colab

Nếu đã push ZIP vào `submissions/` trên fork thì bỏ qua ô sau. Nếu chưa, đổi `UPLOAD_ZIPS = True`, chọn các ZIP có tên đúng mã task; Colab giữ tạm, **không tự push lên fork**. Không upload dữ liệu cá nhân hoặc dữ liệu không được phép đưa lên Colab.

In [ ]:
# Tùy chọn: nếu ZIP CVAT chưa được push lên fork, đổi thành True và chạy ô này.
# Colab chỉ giữ file tạm trong phiên làm việc; bạn vẫn phải upload ZIP lên fork để nộp.
UPLOAD_ZIPS = False
if UPLOAD_ZIPS:
    assert IN_COLAB, "Ô upload này chỉ dành cho Google Colab."
    from google.colab import files
    uploaded = files.upload()
    for filename, data in uploaded.items():
        if filename != Path(filename).name or not filename.endswith(".zip") or Path(filename).stem not in tasks:
            print("Bỏ qua file không đúng mã task:", filename)
            continue
        target = exports / filename
        target.write_bytes(data)
        print("Đã nhận:", target.name)


In [ ]:
for name, info in tasks.items():
    images, classes = expected_for(name, info, root)
    print(f"{name:18} {info['type']:9} {len(images)} ảnh · {len(classes)} class · {info['weight']} điểm")
    print("  ảnh:", ", ".join(sorted(images)))
print("Tổng điểm tối đa:", sum(info["weight"] for info in tasks.values()))


## Xem một ảnh thực

Ảnh bên dưới là ảnh trong task, không có đường biên hay đáp án. Hãy đối chiếu tên file với CVAT khi tạo task. Đừng nộp ảnh này thay mask.

In [ ]:
from IPython.display import display, Image
task_name = "easy_semantic"  # đổi sang một mã task trong bảng trên nếu cần
image_file = sorted((root / "data" / tasks[task_name]["path"] / "images").glob("*.jpg"))[0]
print(task_name, image_file.name)
display(Image(filename=str(image_file), width=760))


## Tự nhắc trước khi vẽ

- Semantic: mỗi pixel thuộc một lớp vùng; không phải một object riêng.
- Instance: mỗi vật đếm được là một mask riêng.
- Panoptic: vừa stuff vừa từng thing.
- Nếu SAM không có, dùng Brush/Polygon. Đọc `CVAT_SETUP.md`; không chờ cài tool mới được làm.

# 02 · QC semantic và ba checkpoint semantic

Xuất `Segmentation mask 1.1` từ CVAT; để ZIP vào `submissions/<mã_task>.zip`. Ô kiểm chỉ kiểm cấu trúc, ảnh và labelmap, **không so đáp án**. Không cần chạy ô này để vẽ được trên CVAT.

In [ ]:
semantic_names = [name for name, info in tasks.items() if info["type"] == "semantic"]
for name in semantic_names:
    result = inspect_task(name, exports / f"{name}.zip", root)
    status = "LỖI" if result["errors"] else ("CHƯA XUẤT" if not Path(result["file"]).exists() else "OK")
    print("\n", name, "·", status)
    print("  ảnh mask:", ", ".join(result["details"].get("mask_images", [])) or "—")
    for note in result["errors"]: print("  SỬA:", note)
    for note in result["warnings"]: print("  KIỂM:", note)


## Xem mask đã export

Chọn đúng task và ảnh. Màu chỉ giúp nhìn vùng; mép road/sidewalk, cột mảnh và lỗ phủ vẫn phải đối chiếu ảnh gốc bằng mắt. Nếu chưa có ZIP, ô sẽ chỉ báo bước cần làm.

In [ ]:
import io, zipfile
from IPython.display import display, Image
task_name = "easy_semantic"
zip_path = exports / f"{task_name}.zip"
if zip_path.is_file():
    with zipfile.ZipFile(zip_path) as archive:
        masks = sorted(n for n in archive.namelist() if "SegmentationClass/" in n and n.endswith(".png"))
        if masks:
            print("Mask:", masks[0])
            display(Image(data=archive.read(masks[0]), width=760))
        else: print("Không có mask SegmentationClass; kiểm lại format export.")
else: print("Chưa có ZIP:", zip_path.name)


## Câu hỏi tự QC

1. Road và sidewalk được phân theo chức năng hay màu ảnh? 2. Có vùng nhìn thấy mà chưa gán class không? 3. Nét mảnh ở `cp3_thin` đã được xem ở mức zoom lớn chưa? Ghi lỗi và hành động sửa vào `REPORT.md`.

# 03 · QC instance và ba checkpoint instance

Xuất `COCO 1.0`. Một object vật lý = một mask. Tự vẽ object Medium đầu trước gợi ý tự động và ghi quy tắc vào `REPORT.md`; gợi ý không thay quyết định của bạn. COCO `annotation_id` không phải mã object bền vững qua hai lần export.

In [ ]:
instance_names = [name for name, info in tasks.items() if info["type"] == "instance"]
for name in instance_names:
    result = inspect_task(name, exports / f"{name}.zip", root)
    status = "LỖI" if result["errors"] else ("CHƯA XUẤT" if not Path(result["file"]).exists() else "OK")
    print("\n", name, "·", status)
    print("  số annotation:", result["details"].get("annotation_count", "—"))
    print("  kiểu mask:", result["details"].get("segmentation_kinds", {}))
    for note in result["errors"]: print("  SỬA:", note)
    for note in result["warnings"]: print("  KIỂM:", note)


## Đếm theo ảnh và class

Bảng này là số mask **bạn đã nộp**, không phải số object đúng. So lại trực quan với từng ảnh trong CVAT để tìm thiếu/thừa, gộp/tách sai, vật bị che vẫn là một object.

In [ ]:
task_name = "medium_instance"  # đổi thành cp1_holes, cp2_slice hoặc cp5_occlusion
result = inspect_task(task_name, exports / f"{task_name}.zip", root)
for key, count in result["details"].get("counts_by_image_class", {}).items():
    print(f"{key}: {count}")
if not result["details"].get("counts_by_image_class"): print("Chưa có object để đếm; kiểm ZIP và format.")


## Ca cần phán đoán

- `cp1_holes`: theo quy tắc task, kính/lỗ nằm trong mask, không tự khoét.
- `cp2_slice`: hai xe cùng lớp sát nhau vẫn là hai instance.
- `cp5_occlusion`: vật bị che thành hai phần nhìn thấy vẫn là một instance.
- Nếu class sai hoặc mask ăn nền, sửa trong CVAT, Save, export lại ZIP.

# 04 · QC panoptic

`hard_panoptic` có hai ảnh, 12 class. Vẽ stuff (road, sky…) và từng thing (car #1, car #2…). Theo hợp đồng starter, export `COCO 1.0`; kiểm này chỉ thấy mask và class trong ZIP, **không chứng minh PQ hay mask đúng**.

In [ ]:
name = "hard_panoptic"
result = inspect_task(name, exports / f"{name}.zip", root)
print("Ảnh:", result["details"].get("images", []))
print("Số mask:", result["details"].get("annotation_count", "—"))
print("Polygon/RLE:", result["details"].get("segmentation_kinds", {}))
for key, count in result["details"].get("counts_by_image_class", {}).items(): print(key, count)
for note in result["errors"]: print("SỬA:", note)
for note in result["warnings"]: print("KIỂM:", note)


## Kiểm bằng mắt trong CVAT trước khi export lại

1. Thing đếm được đã tách từng mask chưa? 2. Stuff có phủ phần thấy được không? 3. Có chồng lấn hoặc vùng chưa phủ ở rìa vật không? 4. Vật bị che: chỉ gán phần nhìn thấy; ghi ca mơ hồ vào report. Công cụ không tự phát hiện đầy đủ các lỗi này.

# 05 · Kiểm trước khi nộp

Bài nộp là **link fork của bạn trên VLearn**. Fork cần `REPORT.md` đã điền và các ZIP CVAT trong `submissions/`. Bạn có thể nộp phần hoàn thành trong 240 phút; task chưa xong cần ghi rõ trong report. Các ô dưới chỉ hỗ trợ tự kiểm, không chấm điểm.

In [ ]:
from inspect_submissions import inspect_all
qc = inspect_all(exports, root)
for row in qc["tasks"]:
    status = "LỖI" if row["errors"] else ("CHƯA CÓ" if not Path(row["file"]).exists() else "OK")
    print(f"{status:7} {row['task']}")
    for error in row["errors"]: print("   !", error)
print("Lỗi hợp đồng:", qc["error_count"], "· task chưa có ZIP:", qc["missing_count"])
print("ZIP tên lạ:", qc["unknown_zips"])


## Gói lưu trữ tùy chọn

`REPORT.md` đã có sẵn ở gốc fork: hãy điền và commit trên GitHub. Ô dưới chỉ tạo thêm ZIP lưu trữ trong phiên notebook, **không phải hình thức nộp**. Bài nộp vẫn là link fork có report và các ZIP riêng trong `submissions/` trên VLearn trong 24 giờ. Nếu ô báo lỗi, sửa trong CVAT và export lại.

In [ ]:
from package_submission import package
learner_id = ""  # điền mã học viên, ví dụ D5_012; không dùng họ tên đầy đủ
if not learner_id:
    print("Điền learner_id rồi chạy lại ô này.")
else:
    output = root / f"day5-{learner_id}.zip"
    try:
        manifest = package(exports, root / "REPORT.md", output, learner_id)
        print("Gói nộp:", output)
        print("Có:", manifest["tasks_present"])
        print("Chưa có:", manifest["tasks_missing"])
    except ValueError as exc:
        print("Chưa thể đóng gói:", exc)


## Sau khi nộp

Mở lại fork trên GitHub, kiểm `REPORT.md` đã điền và ZIP đã xuất hiện trong `submissions/`, rồi nộp link fork trên VLearn. Giữ bản ZIP gốc đến khi nhận phản hồi. Không chỉnh sửa export bên trong ZIP. Điểm 100 chỉ do người chấm đối chiếu reference; QC cấu trúc hoặc hai mask giống nhau không phải điểm.